# SEA Operational Review — 규칙 민감도 (위험 확률 아님)

이 노트북은 **학습 기반 Predict가 아닙니다.** 회사가 정한 업무 기준에 따라 어떤 확인 항목이 켜지는지를 그립니다.

`입력 변화 → Rule → Exception → 담당자 확인`

- 엔진 원본: `src/clocks.ts`, `src/experiments.ts` (TypeScript가 제출 프로토타입의 진실)
- 이 파일: 같은 임계값의 Python 쌍둥이. Colab/Jupyter에서 히트맵을 보기 위함
- 기본값: ETA 검토 창 **6h**, 연결 여유 하한 **24h**
- **하지 않는 것:** 연결 실패 82%, 추가비용 180만원, 추천 1순위

실행 위치: `sea-platform/ai/` 또는 `sea-platform`에서 `python ai/ops_experiments.py`

In [ ]:
# Colab이면 한 번만
# %pip install matplotlib -q

from pathlib import Path
import sys

here = Path.cwd()
if (here / "ops_experiments.py").exists():
    sys.path.insert(0, str(here))
elif (here / "ai" / "ops_experiments.py").exists():
    sys.path.insert(0, str(here / "ai"))

from ops_experiments import ascii_heatmap, extras, label_of, matrix, review_tags, run

payload = run(write_plot=True)
print("R6 Cut-off-from-ETA never fires:", payload["r6NeverFires"])
print(ascii_heatmap(payload["matrix"]))

## 대표 칸 (확률 아님)

| ETA 변경 | 연결 여유 | SEA 확인 항목 |
| ---: | ---: | --- |
| +2h | 30h | 일반 모니터링 |
| +4h | 30h | 일반 모니터링 |
| +8h | 30h | ETA 창 확인 |
| +4h | 20h | 연결항차 확인 |
| +4h | 10h | 연결항차 확인 |
| +8h | 10h | 연결 + ETA 확인 |

추가: 부두 T2→T3 → 내륙 게이트가 같이 켜짐. ETB 미갱신 → 접안 ETB 재확인.

In [ ]:
from IPython.display import Image, display

png = Path("reports/ops_sensitivity_heatmap.png")
if not png.exists():
    png = Path("ai/reports/ops_sensitivity_heatmap.png")
if png.exists():
    display(Image(filename=str(png)))
else:
    print("PNG not found. Re-run the previous cell.")

print("extras")
for row in extras():
    print(f"  {row['case']}: {row['label']}")

print("spot checks")
for eta, slack in [(2, 30), (4, 30), (8, 30), (4, 20), (4, 10), (8, 10)]:
    print(f"  ETA +{eta}h, slack {slack}h -> {label_of(review_tags(eta, slack))}")

## 오류 전파는 TypeScript 엔진에서 측정

Extractor가 잘못 읽으면 확인 항목이 같이 틀어집니다. 시드 실측 (`npx tsx src/eval-run.ts`):

- **IN-NURI** 원문 ETA 07:00 / ETD 20:00. 골드 필드로 넣으면 **변경 없음**. SEA가 ETD를 ETA로 읽으면 **예외 전표 · ETA +13h / 창 6h**.
- **IN-B** 원문 ETA 12:00 / ETB 10:00. 규칙 추출은 **발송 차단**. SEA/하이브리드가 ETB를 놓치면 **예외 전표**가 되어 잠금이 안 걸림.

워크스페이스 **검증** 탭에 같은 표가 라이브로 붙어 있습니다. 필드 F1만 올리는 실험이 아니라, 오추출이 후속 업무 판단으로 전파되는 경로를 검증합니다.